## Simulation Functions

In [ ]:

def normalize_boxscore_variables(df):
    df_ = df.copy()
    # FGM > [36, 45]
    df_[['home_FGM', 'away_FGM']] = (df_[['home_FGM', 'away_FGM']] - 36)/(45 - 36)
    # FGA > [82, 95]
    df_[['home_FGA', 'away_FGA']] = (df_[['home_FGA', 'away_FGA']] - 82)/(95 - 82)
    # FGP > [0.42, 0.51]
    df_[['home_FGP', 'away_FGP']] = (df_[['home_FGP', 'away_FGP']] - 0.42)/(0.51 - 0.42)
    # 3FGM > [8, 17]    
    df_[['home_3FGM', 'away_3FGM']] = (df_[['home_3FGM', 'away_3FGM']] - 8)/(17 - 8)
    # 3FGA > [25, 44]
    df_[['home_3FGA', 'away_3FGA']] = (df_[['home_3FGA', 'away_3FGA']] - 25)/(44 - 25)
    # 3FGP > [0.3, 0.4]
    df_[['home_3FGP', 'away_3FGP']] = (df_[['home_3FGP', 'away_3FGP']] - 0.3)/(0.4 - 0.3)
    # FTM > [13, 23]    
    df_[['home_FTM', 'away_FTM']] = (df_[['home_FTM', 'away_FTM']] - 13)/(23 - 13)
    # FTA > [18, 28]
    df_[['home_FTA', 'away_FTA']] = (df_[['home_FTA', 'away_FTA']] - 18)/(28 - 18)
    # FTP > [0.7, 0.85]    
    df_[['home_FTP', 'away_FTP']] = (df_[['home_FTP', 'away_FTP']] - 0.7)/(0.85 - 0.7)
    # RebOff > [8, 14]    
    df_[['home_RebOff', 'away_RebOff']] = (df_[['home_RebOff', 'away_RebOff']] - 8)/(14 - 8)
    # RebDef > [30, 40]    
    df_[['home_RebDef', 'away_RebDef']] = (df_[['home_RebDef', 'away_RebDef']] - 30)/(40 - 30)
    # RebTot > [40 , 50] 
    df_[['home_RebTot', 'away_RebTot']] = (df_[['home_RebTot', 'away_RebTot']] - 40)/(50 - 40)
    # AST > [20, 30]    
    df_[["home_AST", "away_AST"]] = (df_[["home_AST", "away_AST"]] - 20)/(30 - 20)
    # STL > [5, 10]
    df_[["home_STL", "away_STL"]] = (df_[["home_STL", "away_STL"]] - 5)/(10 - 5)
    # BLK > [3, 7]    
    df_[["home_BLK", "away_BLK"]] = (df_[["home_BLK", "away_BLK"]] - 3)/(7 - 3)
    # TO > [10, 17]    
    df_[["home_TO", "away_TO"]] = (df_[["home_TO", "away_TO"]] - 10)/(17 - 10)
    # FP > [17, 23]    
    df_[["home_FP", "away_FP"]] = (df_[["home_FP", "away_FP"]] - 17)/(23 - 17)
    # Points > [100, 124]
    df_[["home_points", "away_points"]] = (df_[["home_points", "away_points"]] - 100)/(124 - 100)
    # PMP > [-10, 10]
    df_[["home_PMP", "away_PMP"]] = (df_[["home_PMP", "away_PMP"]] - (-10))/(10 - (-10))
    return df_

def put_categorical_training_outcomes(df):
    h, a = 'home_outcome', 'away_outcome'
    return df\
    .assign(ctgc = 0)\
    .assign(ctgc = lambda x: [2 if i == 0 and j >= 0.5 and k >= 0.5 else i for i, j, k in zip(x['ctgc'], x[h], x[a])])\
    .assign(ctgc = lambda x: [3 if i == 0 and j >= 0.5 and k < 0.5 else i for i, j, k in zip(x['ctgc'], x[h], x[a])])\
    .assign(ctgc = lambda x: [4 if i == 0 and j < 0.5 and k >= 0.5 else i for i, j, k in zip(x['ctgc'], x[h], x[a])])\
    .assign(ctgc = lambda x: [5 if i == 0 and j < 0.5 and k < 0.5 else i for i, j, k in zip(x['ctgc'], x[h], x[a])])\
    .assign(ctgc = lambda x: [1 if i >= 0.4 and i <= 0.6 and j >= 0.4 and j <= 0.6 else k for i, j, k in zip(x[h], x[a], x['ctgc'])])

class NGramModel_ExtraInfo(nn.Module):
    """
    A Neural Network designed for N-Gram modeling that incorporates additional 
    numerical input features into the model architecture alongside the standard 
    token embeddings, with optional Layer Normalization and Dropout.

    Attributes:
        embedding (nn.Embedding): Embedding layer to convert token indices into 
        dense vectors.
        flatten (nn.Flatten): Flattens the output of the embedding layer to 
        prepare for linear layers.
        network (nn.Sequential): A sequence of linear and activation layers 
        culminating in the output layer.

    Parameters:
        context_size (int): The number of tokens used for context in embeddings.
        vocab_size (int): The size of the vocabulary in the embedding layer.
        embedding_dim (int): The dimensionality of the embedding vectors.
        numerical_input_dim (int): The dimensionality of the numerical feature vectors.
        num_hidden_neurons (int or list): The number of neurons in each hidden layer.
                                          If a list, specifies neurons for each layer.
        num_layers (int): The total number of hidden layers in the model.
        activation (callable): The activation function used in the network.
        dropout_rate (float): Dropout probability (0.0 means no dropout).
        use_layer_norm (bool): Whether to apply Layer Normalization in the layers.
    """
    def __init__(self, context_size, vocab_size, embedding_dim, numerical_input_dim, 
                 num_hidden_neurons, num_layers, activation = nn.Tanh, dropout_rate = 0.0, use_layer_norm = False):
        super(NGramModel_ExtraInfo, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim=1)

        # Initialize the network layers
        layers = self._init_layers(context_size, vocab_size, embedding_dim, numerical_input_dim, 
                                   num_hidden_neurons, num_layers, activation, dropout_rate, use_layer_norm)

        # Use nn.Sequential to manage the network layers
        self.network = nn.Sequential(*layers)

    def _init_layers(self, context_size, vocab_size, embedding_dim, numerical_input_dim, 
                     num_hidden_neurons, num_layers, activation, dropout_rate, use_layer_norm):
        """
        Initializes the layers of the network based on the provided specifications.
        """
        layers = []
        input_size = embedding_dim * context_size + numerical_input_dim

        # Handle num_hidden_neurons as a list or single value
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        elif len(num_hidden_neurons) != num_layers:
            raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Append the first layer with optional LayerNorm and Dropout
        layers.append(nn.Linear(input_size, num_hidden_neurons[0]))
        if use_layer_norm:
            layers.append(nn.LayerNorm(num_hidden_neurons[0]))
        layers.append(activation())
        layers.append(nn.Dropout(dropout_rate))

        # Append subsequent hidden layers with optional LayerNorm and Dropout
        for i in range(1, num_layers):
            layers.append(nn.Linear(num_hidden_neurons[i - 1], num_hidden_neurons[i]))
            if use_layer_norm:
                layers.append(nn.LayerNorm(num_hidden_neurons[i]))
            layers.append(activation())
            layers.append(nn.Dropout(dropout_rate))

        # Output layer
        layers.append(nn.Linear(num_hidden_neurons[-1], vocab_size))
        return layers

    def forward(self, train_token, train_numerical):
        """
        Forward pass of the model.

        Parameters:
            train_token (torch.Tensor): Tensor of token indices.
            train_numerical (torch.Tensor): Tensor of numerical features.

        Returns:
            torch.Tensor: The output logits from the final layer of the network.
        """
        # Embed and flatten token inputs
        embedded = self.embedding(train_token)
        flattened = self.flatten(embedded)

        # Concatenate embedded tokens with numerical inputs
        concatenated = torch.cat((flattened, train_numerical), dim=1)

        # Pass through the network and return output
        return self.network(concatenated)

class SpentTimeModel(nn.Module):
    """
    A neural network designed to predict time spent based on contextual token 
    information and numerical inputs, with customizable layers, activation functions, 
    optional Layer Normalization, and Dropout.

    Parameters:
        context_size (int): Number of tokens in the context window.
        vocab_size (int): Size of the vocabulary for the embedding layer.
        time_size (int): The output dimension representing time categories or continuous values.
        embedding_dim (int): Dimensionality of the embedding vectors.
        numerical_input_dim (int): Dimensionality of the numerical input features.
        num_hidden_neurons (int or list): Number of neurons in each hidden layer. 
                                          If a list, specifies neurons for each layer.
        num_layers (int): Number of hidden layers in the network.
        activation_function (callable): Activation function to be used in hidden layers.
        dropout_rate (float): Dropout probability (0.0 means no dropout).
        use_layer_norm (bool): Whether to apply Layer Normalization in the layers.
    """

    def __init__(self, context_size, vocab_size, time_size, embedding_dim, numerical_input_dim,
                 num_hidden_neurons, num_layers, activation_function = nn.Tanh, dropout_rate = 0.0, use_layer_norm = False):
        super(SpentTimeModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.flatten = nn.Flatten(start_dim = 1)

        # Building the network dynamically based on the number of layers
        layers = []
        input_features = embedding_dim * context_size + numerical_input_dim

        # Handle num_hidden_neurons as a list or single value
        if isinstance(num_hidden_neurons, int):
            num_hidden_neurons = [num_hidden_neurons] * num_layers
        elif len(num_hidden_neurons) != num_layers:
            raise ValueError("Length of num_hidden_neurons list must match num_layers.")

        # Append the first layer with optional LayerNorm and Dropout
        layers.append(nn.Linear(input_features, num_hidden_neurons[0]))
        if use_layer_norm:
            layers.append(nn.LayerNorm(num_hidden_neurons[0]))
        layers.append(activation_function())
        layers.append(nn.Dropout(dropout_rate))

        # Append additional hidden layers with optional LayerNorm and Dropout
        for i in range(1, num_layers):
            layers.append(nn.Linear(num_hidden_neurons[i - 1], num_hidden_neurons[i]))
            if use_layer_norm:
                layers.append(nn.LayerNorm(num_hidden_neurons[i]))
            layers.append(activation_function())
            layers.append(nn.Dropout(dropout_rate))

        # Output layer to predict the time spent
        layers.append(nn.Linear(num_hidden_neurons[-1], time_size))

        # Combine all layers into a Sequential module
        self.network = nn.Sequential(*layers)

    def forward(self, train_token, train_numerical):
        """
        Defines the forward pass of the model using token indices and numerical features.

        Parameters:
            train_token (torch.Tensor): Tensor of token indices.
            train_numerical (torch.Tensor): Tensor of numerical features.

        Returns:
            torch.Tensor: The output tensor representing predicted time values.
        """
        # Embed and flatten token inputs
        embedded = self.embedding(train_token)
        flattened = self.flatten(embedded)

        # Concatenate embedded tokens with numerical inputs
        concatenated = torch.cat((flattened, train_numerical), dim=1)

        # Pass through the network and return output
        return self.network(concatenated)

def define_special_tokens(config):
    if config == 1:
        # tokens que encerram o periodo
        end_period_tokens = [91, 92, 93, 94, 95, 97, 98, 99, 154, 155, 156, 157, 158, 159, 84, 85, 86, 87, 88, 89, 90]
        # tokens especiais que fazem com que possamos sortear jogadas com o tempo restante = 0
        special_tokens = [ 96,  12,  13,  28,  29,  48,  49,  54,  55,  60,  61,  62,  63, 111, 112, 114, 117, 121, 
                          122, 124, 127, 131, 132, 134, 137, 142, 143, 147, 148, 150, 153, 239, 240, 243, 244, 257, 
                          258, 280, 281, 333, 334]
        # tokens que sao arremessos errados que sao bloqueados e dai necessitam o sorteio de um token de rebote 
        # antes do sorteio de um token de encerramento
        shot_blocked_tokens = [104, 105, 108, 109, 162, 163]
        # define a posse para a equipe mandante no periodo 1
        ball_possession_home_p1 = [86]
        # define a posse para a equipe visitante no periodo 1
        ball_possession_away_p1 = [87]
        # define a posse para nenuma das equipes no periodo 1
        ball_possession_undefined_p1 = [85]
        # define a posse de bola para equipe mandante depois de um primeiro token sem definir no primeiro periodo
        ball_possession_undefined_home_p1 = [66, 68, 70, 72, 78, 86, 511, 518]
        # define a posse de bola para equipe visitante depois de um primeiro token sem definir no primeiro periodo
        ball_possession_undefined_away_p1 = [67, 69, 71, 73, 79, 87, 512, 517]
        # define a posse para nenhuma das equipes no overtime
        ball_possession_undefined_pot = [84]
        # define a posse de bola para equipe mandante depois de um primeiro token sem definir no overtime
        ball_possession_undefined_home_pot = [66, 68]
        # define a posse de bola para equipe visitante depois de um primeiro token sem definir no overtime
        ball_possession_undefined_away_pot = [67, 69]

        home_pts_1 = [164, 168, 172, 177, 181, 189, 191, 195, 199, 203, 209, 213, 217, 219, 223, 227, 231, 235, 268, 
                      352, 358, 384, 388, 396, 402, 406, 410, 417, 418, 431, 436, 437, 445, 447, 451, 459, 500, 525, 
                      527, 534, 536, 547, 549, 550, 552, 554, 563, 565]

        home_pts_2 = [  1,   3,   5,   7,   8,  10,  12,  14,  18,  20,  24,  26,  28,  30,  34,  38,  42,  44,  46,  
                       48,  50,  54,  56, 170, 183, 241, 243, 254, 270, 386, 391, 392, 408, 413, 414, 423, 424, 425, 
                      427, 439, 449, 454, 455, 502, 533]

        home_pts_3 = [16,  22,  32,  36,  40,  52,  58,  60,  62,  64, 185, 441]

        away_pts_1 = [165, 169, 173, 178, 182, 190, 192, 196, 200, 204, 210, 214, 218, 220, 224, 228, 232, 236, 269, 
                      351, 357, 383, 387, 395, 401, 405, 409, 415, 416, 419, 430, 434, 435, 444, 446, 450, 458, 499, 
                      526, 528, 535, 537, 546, 548, 551, 553, 555, 564, 566]

        away_pts_2 = [  2,   4,   6,   9,  11,  13,  15,  19,  21,  25,  27,  29,  31,  35,  39,  43,  45,  47,  49,  
                       51,  55,  57, 171, 184, 242, 244, 255, 271, 385, 389, 390, 407, 411, 412, 420, 421, 422, 426, 
                       438, 448, 452, 453, 501, 532]
        
        away_pts_3 = [17, 23, 33, 37, 41, 53, 59, 61, 63, 65, 186, 440]
        
        home_fouls = [ 15,  17,  19,  21,  23,  31,  33,  35,  37,  39,  41,  51,  53,  57,  59, 139, 161, 253, 255, 
                      263, 265, 267, 269, 271, 365, 367, 368, 371, 372, 373, 377, 378, 381, 383, 385, 387, 389, 390, 
                      393, 395, 397, 399, 401, 403, 405, 407, 409, 411, 412, 415, 416, 419, 420, 421, 422, 426, 428, 
                      430, 432, 434, 435, 438, 440, 442, 444, 446, 448, 450, 452, 453, 456, 458, 460, 462, 464, 476, 
                      478, 480, 481, 483, 486, 488, 490, 492, 494, 496, 497, 499, 501, 503, 505]

        away_fouls = [ 14,  16,  18,  20,  22,  30,  32,  34,  36,  38,  40,  50,  52,  56,  58, 138, 160, 252, 254, 
                      256, 262, 264, 266, 268, 270, 366, 369, 370, 374, 375, 376, 379, 380, 382, 384, 386, 388, 391, 
                      392, 394, 396, 398, 400, 402, 404, 406, 408, 410, 413, 414, 417, 418, 423, 424, 425, 427, 429, 
                      431, 433, 436, 437, 439, 441, 443, 445, 447, 449, 451, 454, 455, 457, 459, 461, 463, 465, 477, 
                      479, 482, 484, 487, 489, 491, 493, 495, 498, 500, 502, 504, 506]
        
        start_period_tokens = [84, 85, 86, 87, 88, 89, 90]
        # tokens permitidos inicio primeiro periodo
        allowed_start_tokens_period1 = [85, 86, 87]
        # tokens permitidos inicio segundo periodo
        allowed_start_tokens_period2 = [88]
        # tokens permitidos inicio terceiro periodo
        allowed_start_tokens_period3 = [89]
        # tokens permitidos inicio quarto periodo
        allowed_start_tokens_period4 = [90]
        # tokens permitidos inicio overtime
        allowed_start_tokens_period5 = [84]
        # tokens permitidos fim periodo 1
        allowed_end_tokens_period1 = [92, 97, 154, 155]
        # tokens permitidos fim periodo 2
        allowed_end_tokens_period2 = [93, 98, 156, 157]
        # tokens permitidos fim periodo 3
        allowed_end_tokens_period3 = [94, 99, 158, 159]
        # tokens permitidos fim periodo 4
        allowed_end_tokens_period4 = [95]
        # tokens permitidos fim overtime
        allowed_end_tokens_overtime = [91]
        # tokens para substituir se forem sorteados ao fim do periodo1
        change_end_tokens_period1_situation1 = [91, 93, 94, 95, 84, 85, 86, 87, 88, 89, 90]
        replace_end_token_period1_situation1 = 92
        change_end_tokens_period1_situation2 = [98, 99]
        replace_end_token_period1_situation2 = 97
        change_end_tokens_period1_situation3 = [156, 158]
        replace_end_token_period1_situation3 = 154
        change_end_tokens_period1_situation4 = [157, 159]
        replace_end_token_period1_situation4 = 155
        # tokens para substituir se forem sorteados ao fim do periodo2
        change_end_tokens_period2_situation1 = [91, 92, 94, 95, 84, 85, 86, 87, 88, 89, 90]
        replace_end_token_period2_situation1 = 93
        change_end_tokens_period2_situation2 = [97, 99]
        replace_end_token_period2_situation2 = 98
        change_end_tokens_period2_situation3 = [154, 158]
        replace_end_token_period2_situation3 = 156
        change_end_tokens_period2_situation4 = [155, 159]
        replace_end_token_period2_situation4 = 157
        # tokens para substituir se forem sorteados ao fim do periodo3
        change_end_tokens_period3_situation1 = [91, 92, 93, 95, 84, 85, 86, 87, 88, 89, 90]
        replace_end_token_period3_situation1 = 94
        change_end_tokens_period3_situation2 = [97, 98]
        replace_end_token_period3_situation2 = 99
        change_end_tokens_period3_situation3 = [154, 156]
        replace_end_token_period3_situation3 = 158
        change_end_tokens_period3_situation4 = [155, 157]
        replace_end_token_period3_situation4 = 159  
        # tokens para substituir se forem sorteados ao fim do periodo4
        change_end_tokens_period4_situation1 = [91, 92, 93, 94, 97, 98, 99, 154, 155, 156, 157, 158, 159, 84, 85, 86, 87, 88, 89, 90]
        replace_end_token_period4_situation1 = 95
        # tokens para substituir se forem sorteados ao fim do overtime
        change_end_tokens_overtime_situation1 = [92, 93, 94, 95, 97, 98, 99, 154, 155, 156, 157, 158, 159, 84, 85, 86, 87, 88, 89, 90]
        replace_end_token_overtime_situation1 = 91
        
        instant_review_token = [96]
        
        dict_special_tokens = {'end_period': end_period_tokens, 'special_plays_end': special_tokens, 'shot_blocked_end': shot_blocked_tokens, 
                               'ballposs_home_p1': ball_possession_home_p1, 'ballposs_away_p1': ball_possession_away_p1, 
                               'ballposs_und_p1': ball_possession_undefined_p1, 'ballposs_und_home_p1': ball_possession_undefined_home_p1, 
                               'ballposs_und_away_p1': ball_possession_undefined_away_p1, 'ballposs_und_pot': ball_possession_undefined_pot, 
                               'ballposs_und_home_pot': ball_possession_undefined_home_pot,
                               'ballposs_und_away_pot': ball_possession_undefined_away_pot,
                               'home_pts1': home_pts_1, 'home_pts2': home_pts_2, 'home_pts3': home_pts_3, 'away_pts1': away_pts_1, 
                               'away_pts2': away_pts_2, 'away_pts3': away_pts_3, 'home_fouls': home_fouls, 'away_fouls': away_fouls,
                               'start_period': start_period_tokens, 'start_period1': allowed_start_tokens_period1, 
                               'start_period2': allowed_start_tokens_period2, 'start_period3': allowed_start_tokens_period3, 
                               'start_period4': allowed_start_tokens_period4, 'start_overtime': allowed_start_tokens_period5, 
                               'end_period1': allowed_end_tokens_period1, 'end_period2': allowed_end_tokens_period2, 
                               'end_period3': allowed_end_tokens_period3, 'end_period4': allowed_end_tokens_period4, 
                               'end_overtime': allowed_end_tokens_overtime,
                               'change_end_p1_s1': change_end_tokens_period1_situation1, 'sub_end_p1_s1': replace_end_token_period1_situation1,
                               'change_end_p1_s2': change_end_tokens_period1_situation2, 'sub_end_p1_s2': replace_end_token_period1_situation2,
                               'change_end_p1_s3': change_end_tokens_period1_situation3, 'sub_end_p1_s3': replace_end_token_period1_situation3,
                               'change_end_p1_s4': change_end_tokens_period1_situation4, 'sub_end_p1_s4': replace_end_token_period1_situation4,
                               'change_end_p2_s1': change_end_tokens_period2_situation1, 'sub_end_p2_s1': replace_end_token_period2_situation1,
                               'change_end_p2_s2': change_end_tokens_period2_situation2, 'sub_end_p2_s2': replace_end_token_period2_situation2,
                               'change_end_p2_s3': change_end_tokens_period2_situation3, 'sub_end_p2_s3': replace_end_token_period2_situation3,
                               'change_end_p2_s4': change_end_tokens_period2_situation4, 'sub_end_p2_s4': replace_end_token_period2_situation4,                               
                               'change_end_p3_s1': change_end_tokens_period3_situation1, 'sub_end_p3_s1': replace_end_token_period3_situation1,
                               'change_end_p3_s2': change_end_tokens_period3_situation2, 'sub_end_p3_s2': replace_end_token_period3_situation2,
                               'change_end_p3_s3': change_end_tokens_period3_situation3, 'sub_end_p3_s3': replace_end_token_period3_situation3,
                               'change_end_p3_s4': change_end_tokens_period3_situation4, 'sub_end_p3_s4': replace_end_token_period3_situation4,
                               'change_end_p4_s1': change_end_tokens_period4_situation1, 'sub_end_p4_s1': replace_end_token_period4_situation1,
                               'change_end_ot_s1': change_end_tokens_overtime_situation1,'sub_end_ot_s1': replace_end_token_overtime_situation1,
                               'instant_replay': instant_review_token}
                
        return dict_special_tokens
    
def predict_model(model, token_vector, numerical_vector, k = 0, only_consider = None, extra_info = True, type_ = 'tokens'):
    # Usada para sortear a proxima jogada ou o proximo tempo gasto
    # Atualizando o tipo dos vetores e a dimensao, se necessario.
    if isinstance(token_vector, list):
        token_vector = np.array(token_vector, dtype = "int32")
    if isinstance(numerical_vector, list):
        numerical_vector = np.array(numerical_vector)
    if len(token_vector.shape) == 1:
        token_vector = token_vector.reshape(1, -1)
        numerical_vector = numerical_vector.reshape(1, -1)

    if not isinstance(token_vector, torch.Tensor):
        token_vector = torch.tensor(token_vector)
    if not isinstance(numerical_vector, torch.Tensor):
        numerical_vector = torch.tensor(numerical_vector, dtype = torch.float32)
        
    # gerando os log-valores para os tokens de jogada ou tempo a partir dos vetores de input.
    logits = model(token_vector, numerical_vector)

    # se for usada para sortear a proxima jogada, nao permite que o token 0 seja sorteado (token de inicializacao)
    if type_ == 'tokens': logits[:, 0] = -float("inf")

    # se quisermos sortear uma jogada apenas considerando alguns tokens, então zera a probabilidade dos que não estão sendo considerados
    if only_consider is not None:
        logits[:, ~torch.isin(torch.arange(logits.size(1)), torch.tensor(only_consider))] = -float('inf')

    # transforma tudo em probabilidade
    probabilities = torch.softmax(logits, dim = 1)

    # Se for necessario ter uma lista das probabilidades que o modelo atribui a cada token, k precisa ser maior que 0.
    if k > 0:
        topk = torch.topk(probabilities, k, dim = 1)
        topk_preds = {'indices': list(topk.indices.detach().numpy()[0]), 'probs': list(topk.values.detach().numpy()[0])}
        return pd.DataFrame(topk_preds) #topk_preds 
    else:
        # se a funcao foi chamada com apenas um vetor de input entao so retorna uma jogada, caso contrario, retorna jogadas para todos os vetores de input
        if token_vector.shape[0] == 1:
            sorted_item = torch.multinomial(input = probabilities, num_samples = 1).item()
            sorted_prob = probabilities[0, sorted_item].item()
            # se solicitamos extra_info, entao além da jogada tambem retorna a probabilidade associada a essa jogada
            if extra_info: return [sorted_item], [sorted_prob]
            return [sorted_item]
        else:
            sorted_items = [torch.multinomial(i, 1).item() for i in probabilities]
            sorted_probs = [probabilities[i, j].item() for i, j in enumerate(sorted_items)]
            if extra_info: return sorted_items, sorted_probs
            return sorted_items
        
def predict_model(model, token_vector, numerical_vector, k = 0, constant_value = 1, only_consider = None, extra_info = True, type_ = 'tokens'):
    # Usada para sortear a proxima jogada ou o proximo tempo gasto
    # Atualizando o tipo dos vetores e a dimensao, se necessario.
    if isinstance(token_vector, list):
        token_vector = np.array(token_vector, dtype = "int32")
    if isinstance(numerical_vector, list):
        numerical_vector = np.array(numerical_vector)
    if len(token_vector.shape) == 1:
        token_vector = token_vector.reshape(1, -1)
        numerical_vector = numerical_vector.reshape(1, -1)

    if not isinstance(token_vector, torch.Tensor):
        token_vector = torch.tensor(token_vector)
    if not isinstance(numerical_vector, torch.Tensor):
        numerical_vector = torch.tensor(numerical_vector, dtype = torch.float32)
        
    # gerando os log-valores para os tokens de jogada ou tempo a partir dos vetores de input.
    if type_ == 'tokens':
        logits = model(token_vector, constant_value * numerical_vector)
    else:
        logits = model(token_vector, numerical_vector)

    # se for usada para sortear a proxima jogada, nao permite que o token 0 seja sorteado (token de inicializacao)
    if type_ == 'tokens': logits[:, 0] = -float("inf")

    # se quisermos sortear uma jogada apenas considerando alguns tokens, então zera a probabilidade dos que não estão sendo considerados
    if only_consider is not None:
        logits[:, ~torch.isin(torch.arange(logits.size(1)), torch.tensor(only_consider))] = -float('inf')

    # transforma tudo em probabilidade
    probabilities = torch.softmax(logits, dim = 1)

    # Se for necessario ter uma lista das probabilidades que o modelo atribui a cada token, k precisa ser maior que 0.
    if k > 0:
        topk = torch.topk(probabilities, k, dim = 1)
        topk_preds = {'indices': list(topk.indices.detach().numpy()[0]), 'probs': list(topk.values.detach().numpy()[0])}
        return pd.DataFrame(topk_preds) #topk_preds 
    else:
        # se a funcao foi chamada com apenas um vetor de input entao so retorna uma jogada, caso contrario, retorna jogadas para todos os vetores de input
        if token_vector.shape[0] == 1:
            sorted_item = torch.multinomial(input = probabilities, num_samples = 1).item()
            sorted_prob = probabilities[0, sorted_item].item()
            # se solicitamos extra_info, entao além da jogada tambem retorna a probabilidade associada a essa jogada
            if extra_info: return [sorted_item], [sorted_prob]
            return [sorted_item]
        else:
            sorted_items = [torch.multinomial(i, 1).item() for i in probabilities]
            sorted_probs = [probabilities[i, j].item() for i, j in enumerate(sorted_items)]
            if extra_info: return sorted_items, sorted_probs
            return sorted_items
        
def define_ball_possession(token, last_token, period, ball_possession):
    # assinala qual equipe conseguiu a primeira posse de bola do jogo e do primeiro overtime se ele existir
    dict_tokens = define_special_tokens(config = 1)
    if period == 1:
        if token in dict_tokens['ballposs_home_p1']: return 1
        if token in dict_tokens['ballposs_away_p1']: return -1
        if last_token in dict_tokens['ballposs_und_p1']:
            if token in dict_tokens['ballposs_und_home_p1']: return 1
            if token in dict_tokens['ballposs_und_away_p1']: return -1
    if period >= 5:
        if last_token in dict_tokens['ballposs_und_pot']:
            if token in dict_tokens['ballposs_und_home_pot']: return 1
            if token in dict_tokens['ballposs_und_away_pot']: return -1
    return ball_possession

def define_game_fouls(period):
    # Define o limite de faltas a depender do periodo
    foul_limit = 4 if period <= 4 else 3
    # Retorna uma tupla com o mesmo valor de limite de faltas para o mandante e o visitante
    return foul_limit, foul_limit

def create_game_status(random_play):
    # Inicia o contexto do jogo (pontos, faltas, posse de bola e tempo restante apos o sorteio da primeira jogada do primeiro periodo)
    game_status = {}
    game_status['home_points'], game_status['away_points'] = 0, 0
    game_status['limhteam'], game_status['limvteam'] = define_game_fouls(1)
    game_status['ball_possession'] = define_ball_possession(token = random_play, last_token = 0, period = 1, ball_possession = 0)
    game_status['remaining_time'] = 7200
    game_status['period'] = 0
    return game_status

def game_log_to_dataframe(game_log, dict_integer_to_token = integer_to_token):
    df_game_log = pd.concat(
        objs = [pd.DataFrame(game_log).query("period.notna()"), 
                pd.DataFrame(list(filter(lambda x: x is not None, game_log['game_variables'])))\
                .drop(columns = 'period')
               ], axis = 1)\
    .drop(columns = ['index', 'game_variables'])\
    .assign(indice = lambda df: [dict_integer_to_token[i] for i in df['token']])\
    .assign(token_variables = lambda df: df['token_variables'].shift(1),
            numerical_variables = lambda df: df['numerical_variables'].shift(1))\
    .assign(token_variables = lambda df: df['token_variables'].fillna(0),
            numerical_variables = lambda df: df['numerical_variables'].fillna(0))

    columns_df_game_log = ['period', 'token', 'indice', 'prob', 'time', 'home_points', 
                           'away_points', 'limhteam', 'limvteam', 'remaining_time', 
                           'ball_possession', 'token_variables', 'numerical_variables']
    
    return df_game_log[columns_df_game_log]

def filter_elements(element, indices):
    # filtrando elementos em uma lista a partir das posicoes.
    return [k for i, k in enumerate(element) if i in indices]

def get_element_from_dict(dict_, key):
    #get the values in a dicionary passing the key.
    return [element[key] for element in dict_]

def concatenate_element_in_game_log(game_log, indices, key, new_value):
    # Procedimento que concatena (in-place) um elemento em uma lista onde cada 
    # elemento e um dicionario. Acessa a chave 'index' de cada dicionario guardado 
    # na lista. Esse valor guarda quantas jogadas foram geradas para cada simulacao.  
    list_index = get_element_from_dict(dict_ = game_log, key = 'index')
    if isinstance(new_value, list):
        if len(indices) != len(new_value):
            print(indices, new_value)
            raise ValueError("Length of 'indices' and 'new_value' must be the same when 'new_value' is a list.")
        
        # Update each specified index with the corresponding new value
        for idx, val in zip(indices, new_value):
            if 0 <= idx < len(game_log):
                if key != 'index':
                    game_log[idx][key][list_index[idx]] = val
                else:
                    game_log[idx][key] = val
            else:
                print(f"Warning: Index {idx} is out of bounds.")
    else:
        # If new_value is a single value, apply it to all specified indices
        for idx in indices:
            if 0 <= idx < len(game_log):
                if key != 'index':
                    game_log[idx][key][list_index[idx]] = new_value
                else:
                    game_log[idx][key] = new_value
            else:
                print(f"Warning: Index {idx} is out of bounds.")

def update_game_log_by_indices(game_log, period, play, prob, time, game_status, token_vector, numerical_vector, indices):
    
    list_token_vector = [token_vector[i] for i in range(token_vector.shape[0])]
    list_numerical_vector = [numerical_vector[i] for i in range(numerical_vector.shape[0])]
    list_index = [i + 1 for i in get_element_from_dict(dict_ = filter_elements(element = game_log, indices = indices), key = 'index')]

    concatenate_element_in_game_log(game_log, indices, 'period', period)
    concatenate_element_in_game_log(game_log, indices, 'token', play)
    concatenate_element_in_game_log(game_log, indices, 'prob', prob)
    concatenate_element_in_game_log(game_log, indices, 'time', time)
    concatenate_element_in_game_log(game_log, indices, 'game_variables', game_status)
    concatenate_element_in_game_log(game_log, indices, 'token_variables', list_token_vector)
    concatenate_element_in_game_log(game_log, indices, 'numerical_variables', list_numerical_vector)
    concatenate_element_in_game_log(game_log, indices, 'index', list_index)
    return game_log

def update_game_log(game_log, period, play, prob, time, game_status, token_vector, numerical_vector, keep_generating):
    # garantindo que o numerical_vector nao sera uma lista.
    if isinstance(numerical_vector, list): numerical_vector = np.array(numerical_vector)

    # Coletando as simulacoes que devemos atualizar o game_log
    idx_to_update_game_log = [i for i, j in enumerate(keep_generating) if j > 0 and j < 6]
    
    # Mantendo nos valores sorteados somente aqueles que sao das simulacoes que precisam ser sorteadas.
    play = [play_ for simulation, play_ in enumerate(play) if simulation in idx_to_update_game_log]
    prob = [prob_ for simulation, prob_ in enumerate(prob) if simulation in idx_to_update_game_log]
    time = [time_ for simulation, time_ in enumerate(time) if simulation in idx_to_update_game_log] 
    
    # Atualizando o game_log para as simulacoes necessarias.
    update_game_log_by_indices(game_log = game_log, period = period, play = play, prob = prob, time = time, 
                               game_status = filter_elements(element = game_status, indices = idx_to_update_game_log), 
                               token_vector = token_vector[idx_to_update_game_log], 
                               numerical_vector = numerical_vector[idx_to_update_game_log], 
                               indices = idx_to_update_game_log)
    
    # Atualizando a lista keep_generating. Onde era 2 ira ser 0. Isso significa que as simulacoes onde uma 
    # jogada de encerramento foi sorteada, a simulacao tambem precisa ser encerrada.
    keep_generating = [0 if i == 2 else i for i in keep_generating]

    # Atualizando a lista keep_generating. Onde era 3 ira ser 6. Isso significa que as simulacoes onde uma
    # jogada pertencente a uma lista de tokens especiais foi sorteada apos o tempo permitido, essa jogada
    # estara no game_log, mas a proxima tera que ser uma jogada de encerramento.
    keep_generating = [6 if i == 3 else i for i in keep_generating]

    # Atualizando a lista keep_generating. Onde era 5 ira ser 6. Isso significa que nessas simulacoes a
    # jogada anteriormente sorteada estava em uma segunda lista de tokens especiais, o sorteio dessa jogada 
    # foi sorteada livremente e estarao no game_log mas a proxima tera que ser uma jogada de encerramento.
    keep_generating = [6 if i == 5 else i for i in keep_generating]

    # Caso 4: um token de bloqueio foi soreaado, entao o proximo sorteio de 
    # token (n + 1) é livre mas o (n + 2) precisa ser um encerramento de periodo 

    # Atualizando a lista keep_generating. Onde era 4 ira ser 5. Isso significa que nessas simulacoes a
    # jogada sorteada estava em uma segunda lista de tokens especiais, o sorteio da proxima jogada sera feito
    # livremente e estarao no game_log mas depois disso tera que ser sorteada uma jogada de encerramento.
    keep_generating = [5 if i == 4 else i for i in keep_generating]

    return keep_generating

def update_element_in_game_status(game_status, indices, key, new_value):
    # procedimento que atualiza (in-place) uma lista onde cada elemento e um 
    # dicionario a partir de chave e posicao.
    if isinstance(new_value, list):
        if len(indices) != len(new_value):
            raise ValueError("Length of 'indices' and 'new_value' must be the same when 'new_value' is a list.")
        for idx, val in zip(indices, new_value):
            if 0 <= idx < len(game_status):
                game_status[idx][key] = val
            else:
                print(f"Warning: Index {idx} is out of bounds.")
    else:
        # If new_value is a single value, apply it to all specified indices
        for idx in indices:
            if 0 <= idx < len(game_status):
                game_status[idx][key] = new_value
            else:
                print(f"Warning: Index {idx} is out of bounds.")

def update_numerical_variables(game_status, period):
    # Create a period list with a 1 in the correct position
    period_list = [0, 0, 0, 0, 0]
    if period > 0: period_list[min(period, 5) - 1] = 1
    # Precompute remaining time ratio
    remaining_time_ratio = 1 - max(0, game_status['remaining_time'] / 7200)
    # Foul variables
    foul_vars = [int(game_status['limhteam'] < 1), int(game_status['limvteam'] < 1)]
    # Points difference normalized to [-1, 1]
    points_diff = (game_status['home_points'] - game_status['away_points']) / 30
    points_vars = [max(-1, min(1, points_diff))]
    # Time variables
    remaining_time = game_status['remaining_time']
    time_vars = [int(remaining_time <= 600), int(remaining_time <= 200), int(remaining_time <= 100)]

    # Leading variable
    hpts, apts = game_status['home_points'], game_status['away_points']
    leading_var = [1 if hpts > apts else -1 if hpts < apts else 0]

    # Ball possession variable
    ball_possession_var = [game_status['ball_possession']]

    # Return the combined numerical vector
    values = period_list + [remaining_time_ratio] + foul_vars + points_vars + time_vars + leading_var + ball_possession_var
    numerical_variables = values
    return numerical_variables

def update_variables(random_play, tokens, game_status, period):
    game_status = [dict(item) for item in game_status]
    # atualiza o vetor de tokens e as variaveis numericas para gerar a proxima jogada ou proximo tempo gasto

    #_tokens = np.concatenate([np.array(tokens)[:, 1:], np.array(random_play, ndmin = 2).T], axis = 1)
    #_tokens1 = np.array([old_vector[1:] + [element] for old_vector, element in zip(tokens, random_play)])
    _tokens = np.array([np.append(old_vector[1:], element) for old_vector, element in zip(tokens, random_play)])

    #numerical_variables sempre sai como array
    numerical_variables = [update_numerical_variables(i, period) for i in game_status]
    return _tokens, np.array(numerical_variables)

def update_game_status(game_status, random_play):
    # atualiza o contexto do jogo a depender da jogada sorteada (atualiza os pontos e o limite de faltas)
    dict_tokens = define_special_tokens(config = 1)
    temp_game_status = [dict(item) for item in game_status]

    list_indices = [i for i in range(len(temp_game_status))]
    list_limhteam = get_element_from_dict(dict_ = temp_game_status, key = 'limhteam')
    list_limvteam = get_element_from_dict(dict_ = temp_game_status, key = 'limvteam')
    list_homepts = get_element_from_dict(dict_ = temp_game_status, key = 'home_points')
    list_awaypts = get_element_from_dict(dict_ = temp_game_status, key = 'away_points')
    list_ballpossession = get_element_from_dict(dict_ = temp_game_status, key = 'ball_possession')

    for i, p in enumerate(random_play):
        if p in dict_tokens['home_pts1']: list_homepts[i] += 1
        if p in dict_tokens['home_pts2']: list_homepts[i] += 2
        if p in dict_tokens['home_pts3']: list_homepts[i] += 3
        if p in dict_tokens['home_fouls']: list_limhteam[i] += -1
        if p in dict_tokens['away_pts1']: list_awaypts[i] += 1
        if p in dict_tokens['away_pts2']: list_awaypts[i] += 2
        if p in dict_tokens['away_pts3']: list_awaypts[i] += 3
        if p in dict_tokens['away_fouls']: list_limvteam[i] += -1        
        if list_ballpossession[i] == 0 and p in dict_tokens['ballposs_und_home_p1']: list_ballpossession[i] = 1
        if list_ballpossession[i] == 0 and p in dict_tokens['ballposs_und_away_p1']: list_ballpossession[i] = -1
                
    update_element_in_game_status(temp_game_status, list_indices, 'limhteam', list_limhteam)
    update_element_in_game_status(temp_game_status, list_indices, 'limvteam', list_limvteam)
    update_element_in_game_status(temp_game_status, list_indices, 'home_points', list_homepts)
    update_element_in_game_status(temp_game_status, list_indices, 'away_points', list_awaypts)
    update_element_in_game_status(temp_game_status, list_indices, 'ball_possession', list_ballpossession)

    return temp_game_status

def fix_last_token(token, period):
    dict_tokens = define_special_tokens(config = 1)
    if period == 1:
        token = dict_tokens['sub_end_p1_s1'] if token in dict_tokens['change_end_p1_s1'] else token
        token = dict_tokens['sub_end_p1_s2'] if token in dict_tokens['change_end_p1_s2'] else token
        token = dict_tokens['sub_end_p1_s3'] if token in dict_tokens['change_end_p1_s3'] else token
        token = dict_tokens['sub_end_p1_s4'] if token in dict_tokens['change_end_p1_s4'] else token
    elif period == 2:
        token = dict_tokens['sub_end_p2_s1'] if token in dict_tokens['change_end_p2_s1'] else token
        token = dict_tokens['sub_end_p2_s2'] if token in dict_tokens['change_end_p2_s2'] else token
        token = dict_tokens['sub_end_p2_s3'] if token in dict_tokens['change_end_p2_s3'] else token
        token = dict_tokens['sub_end_p2_s4'] if token in dict_tokens['change_end_p2_s4'] else token
    elif period == 3:
        token = dict_tokens['sub_end_p3_s1'] if token in dict_tokens['change_end_p3_s1'] else token
        token = dict_tokens['sub_end_p3_s2'] if token in dict_tokens['change_end_p3_s2'] else token
        token = dict_tokens['sub_end_p3_s3'] if token in dict_tokens['change_end_p3_s3'] else token
        token = dict_tokens['sub_end_p3_s4'] if token in dict_tokens['change_end_p3_s4'] else token        
    elif period == 4:
        token = dict_tokens['sub_end_p4_s1'] if token in dict_tokens['change_end_p4_s1'] else token
    else:
        token = dict_tokens['sub_end_ot_s1'] if token in dict_tokens['change_end_ot_s1'] else token
    return token

def random_play_loop(token_vector, numerical_vector, game_status, period, token_model, time_model, match_vector = None):
    dict_tokens = define_special_tokens(config = 1)
    list_end_period_tokens = dict_tokens['end_period']
    temp_game_status = [dict(item) for item in game_status]
    if match_vector is None:
        numerical_vector_ = numerical_vector
    else:
        numerical_vector_ = np.hstack([numerical_vector, match_vector])
        
    # token_vector e numerical_vector precisam ser arrays sorteia a jogada
    random_token, prob_random_token = predict_model(model = token_model, token_vector = token_vector, 
                                                    numerical_vector = numerical_vector_)
    
    # CORRECAO 1: token de encerramento sorteado faltando muito tempo a jogar.
    # verificando se preciso sortear novamente se a jogada sorteada encerra o periodo mas ainda existe 
    # mais de 15 segundos a jogar, entao obrigatoriamente precisa sortear outra que nao encerre o periodo
    list_remaining_time = get_element_from_dict(dict_ = game_status, key = 'remaining_time')
    invalid_play_idx = [i for i, play in enumerate(random_token) if (play in list_end_period_tokens and list_remaining_time[i] > 150)]
    while len(invalid_play_idx) > 0:
        if len(invalid_play_idx) == 1: invalid_play_idx = invalid_play_idx[0]

        temp_random_token, temp_prob_random_token = predict_model(model = token_model, token_vector = token_vector[invalid_play_idx],
                                                                  numerical_vector = numerical_vector_[invalid_play_idx])
        
        if not isinstance(invalid_play_idx, list): invalid_play_idx = [invalid_play_idx]

        random_token = [temp_random_token[invalid_play_idx.index(i)] if i in invalid_play_idx else play for i, play in enumerate(random_token)]
        prob_random_token = [temp_prob_random_token[invalid_play_idx.index(i)] if i in invalid_play_idx else prob for i, prob in enumerate(prob_random_token)]
        
        invalid_play_idx = [i for i, play in enumerate(random_token) if (play in list_end_period_tokens and list_remaining_time[i] > 150)]
        
    # CORRECAO 2: caso o token de encerramento foi sorteado, certifica que ele se refere ao periodo jogado.
    # Corrige o token sorteado se ele encerra o periodo e caso possa estar se referindo ao periodo incorreto
    random_token = [fix_last_token(token = p, period = period) if p in list_end_period_tokens else p for p in random_token]
    # atualiza o status do jogo independente do token sortado
    temp_game_status = update_game_status(game_status = temp_game_status, random_play = random_token)

    # atualiza os vetores de input. Considerando as novas contagens de pontos e faltas, mas o tempo restante antigo
    token_vector, numerical_vector = update_variables(random_play = random_token, tokens = token_vector, game_status = temp_game_status, period = period)
    
    # definindo os tokens que um dos tokens de encerramento foi sorteado. Nesses não precisamos sortear tempo gasto pela jogada.
    list_remaining_time_zero = [i for i, play in enumerate(random_token) if play in list_end_period_tokens]
    if len(list_remaining_time_zero) > 0:
        # definindo o tempo restante do periodo para esses casos igual a zero. O sorteio para essas simulacoes esta encerrado nesse periodo.
        temp_game_status_end_period = filter_elements(element = temp_game_status, indices = list_remaining_time_zero)
        list_former_remaining_time = get_element_from_dict(dict_ = temp_game_status_end_period, key = 'remaining_time')
        update_element_in_game_status(game_status = temp_game_status, indices = list_remaining_time_zero, key = 'remaining_time', new_value = 0)

    # Verificando quais sao as simulacoes que precisam de sorteio de tempo
    idx_random_remaining_time = [i for i, play in enumerate(random_token) if play not in list_end_period_tokens]
    # usa as variaveis de contexto atualizadas apos a geracao da jogada para gerar o tempo gasto
    if match_vector is None:
        numerical_vector_ = numerical_vector
    else:
        numerical_vector_ = np.hstack([numerical_vector, match_vector])
        
    random_time = predict_model(model = time_model,  token_vector = token_vector[idx_random_remaining_time], 
                                numerical_vector = numerical_vector_[idx_random_remaining_time], extra_info = False, 
                                type_ = 'time')

    # CORRECAO 3: se o tempo sorteado for maior que o tempo a jogar, entao precisamos tentar corrigir isso.
    # se o tempo sorteado for maior que o tempo restante, tente por três vezes sortear um tempo válido. Se não conseguir,
    # então deixe o tempo sorteado invalido seguir porque a jogada sera invalidada ou corrigiremos o tempo gasto em outra funcao
    temp_game_status_fix_time = filter_elements(element = temp_game_status, indices = idx_random_remaining_time)
    list_remaining_time = get_element_from_dict(dict_ = temp_game_status_fix_time, key = 'remaining_time')
    idx_invalid_time = [idx for idx, t, rmng_t in zip(idx_random_remaining_time, random_time, list_remaining_time) if t > rmng_t]

    invalid_time_flag = 0
    while len(idx_invalid_time) > 0 and invalid_time_flag < 3:
        if len(idx_invalid_time) == 1: idx_invalid_time = idx_invalid_time[0]

        temp_random_time = predict_model(model = time_model, token_vector = token_vector[idx_invalid_time],
                                         numerical_vector = numerical_vector_[idx_invalid_time], extra_info = False,
                                         type_= 'time')

        if not isinstance(idx_invalid_time, list): idx_invalid_time = [idx_invalid_time]

        random_time = [temp_random_time[idx_invalid_time.index(i)] if i in idx_invalid_time else time for i, time in enumerate(random_time)]
        idx_invalid_time = [i for i, t, rmng_t in zip(idx_random_remaining_time, random_time, list_remaining_time) if t > rmng_t]
        invalid_time_flag += 1

    # CORRECAO 4: se o tempo sorteado contem decimais, mas ainda nao estamos no ultimo minuto do periodo, entao corrija.
    random_time = [10 * (t//10) if (rmng_t - t) > 600 else t for rmng_t, t in zip(list_remaining_time, random_time)]
    
    # CORRECAO 5: se o tempo sorteado deixa o relogio nos ultimos dois minutos do periodo, o limite de faltas precisa ser atualizado.
    # Flag to track if the 2-minute rule has been triggered
    idx_two_final_minutes = [i for i, rmng_t, t in zip(idx_random_remaining_time, list_remaining_time, random_time) if (rmng_t - t) <= 1200]

    # Atualiza o limite de faltas para as duas equipes
    # se a jogada sorteada nao é uma falta (limite de faltas nao mudou entre o contexto do jogo inputado e o contexto do jogo atualizado) entao muda a variavel para ser no máximo 2
    # se a jogada sorteada foi uma falta e o tempo sorteado atribui essa jogada aos 2 minutos finais entao significa que o time já gastou uma das faltas que tinha direito. 
    # Daí muda o contexto do jogo apos a jogada sorteada para ser no máximo 1   
    if len(idx_two_final_minutes) > 0:
        new_game_status = filter_elements(element = temp_game_status, indices = idx_two_final_minutes)
        old_game_status = filter_elements(element = game_status, indices = idx_two_final_minutes)

        updated_home_limit = get_element_from_dict(dict_ = new_game_status, key = 'limhteam')
        former_home_limit  = get_element_from_dict(dict_ = old_game_status, key = 'limhteam')
        updated_home_limit = [min(1, i) if i == j else min(0, i) for i, j in zip(updated_home_limit, former_home_limit)]

        updated_away_limit = get_element_from_dict(dict_ = new_game_status, key = 'limvteam')
        former_away_limit  = get_element_from_dict(dict_ = old_game_status, key = 'limvteam')
        updated_away_limit = [min(1, i) if i == j else min(0, i) for i, j in zip(updated_away_limit, former_away_limit)]

        update_element_in_game_status(game_status = temp_game_status, indices = idx_two_final_minutes, key = 'limhteam', new_value = updated_home_limit)
        update_element_in_game_status(game_status = temp_game_status, indices = idx_two_final_minutes, key = 'limvteam', new_value = updated_away_limit)

    # finalmente atualizando o tempo restante no game_status
    new_remaining_time = [rmng_t - t for rmng_t, t in zip(list_remaining_time, random_time)]
    update_element_in_game_status(game_status = temp_game_status, indices = idx_random_remaining_time, 
                                  key = 'remaining_time', new_value = new_remaining_time)

    # atualizando o vetor numerico de input com o novo tempo restante.
    numerical_vector = [update_numerical_variables(gamestatus, period) for gamestatus in temp_game_status]
    numerical_vector = np.array(numerical_vector)

    # concatenando os tempos gastos das jogadas: para aqueles onde foi necessario sorteio e para aqueles que foi necessario apenas saber o tempo restante.
    if len(list_remaining_time_zero) > 0:
        combined_indices = list_remaining_time_zero + idx_random_remaining_time
        combined_values = list_former_remaining_time + random_time
        # Create a final list using list comprehension
        list_remaining_time = [combined_values[combined_indices.index(i)] if i in combined_indices else None for i in range(max(combined_indices) + 1)]
    else:
        list_remaining_time = random_time
    return random_token, prob_random_token, list_remaining_time, token_vector, numerical_vector, temp_game_status

def random_first_play(token_vector, numerical_vector, period, game_status = None, token_model = None, match_vector = None):
    # Gerando a primeira jogada do periodo
    dict_tokens = define_special_tokens(config = 1)
    if match_vector is None:
        numerical_vector_ = numerical_vector
    else:
        numerical_vector_ = np.hstack([numerical_vector, match_vector])
            
    if period == 1:
        # se for a primeira jogada do jogo, entao é preciso criar o contexto do jogo também
        random_play, prob_random_play = predict_model(model = token_model, token_vector = token_vector,
                                                      numerical_vector = numerical_vector_, only_consider = dict_tokens['start_period1'])
        game_status = [create_game_status(random_play = play) for play in random_play]

        # a partir dos valores sorteados, atualiza o game_status
        token_vector, numerical_vector = update_variables(random_play = random_play, tokens = token_vector,
                                                          game_status = game_status, period = period)

        # sempre a primeira jogada do primeiro periodo "gasta" tempo zero
        random_time = [0] * len(random_play)
        return random_play, prob_random_play, random_time, token_vector, numerical_vector, game_status
    else:
        # se estamos no inicio de qualquer outro periodo, entao precisamos usar o contexto do jogo atual para considerar a pontuacao das equipes
        if period == 2: tokens_list = dict_tokens['start_period2'] * token_vector.shape[0]
        elif period == 3: tokens_list = dict_tokens['start_period3'] * token_vector.shape[0]
        elif period == 4: tokens_list = dict_tokens['start_period4'] * token_vector.shape[0]
        else: tokens_list = dict_tokens['start_overtime'] * token_vector.shape[0]

        # sorteia a primeira jogada de qualquer periodo que nao o primeiro
        random_play, prob_random_play = predict_model(model = token_model, token_vector = token_vector,
                                                      numerical_vector = numerical_vector_, only_consider = tokens_list)

        list_idx = [i for i in range(len(random_play))]
        period_duration = 7200 if period <= 4 else 3000
        allowed_fouls_home, allowed_fouls_away = define_game_fouls(period)
        update_element_in_game_status(game_status = game_status, indices = list_idx, key = 'period', new_value = period)
        update_element_in_game_status(game_status = game_status, indices = list_idx, key = 'limhteam', new_value = allowed_fouls_home)
        update_element_in_game_status(game_status = game_status, indices = list_idx, key = 'limvteam', new_value = allowed_fouls_away)
        update_element_in_game_status(game_status = game_status, indices = list_idx, key = 'remaining_time', new_value = period_duration)

        if period == 5:
            update_element_in_game_status(game_status = game_status, indices = list_idx, key = 'ball_possession', new_value = 0)
    
        # a partir dos valores sorteados, atualiza o game_status
        token_vector, numerical_vector = update_variables(random_play = random_play, tokens = token_vector, 
                                                          game_status = game_status, period = period)

        # sempre a primeira jogada do primeiro periodo "gasta" tempo zero
        random_time = [0] * len(random_play)
        return random_play, prob_random_play, random_time, token_vector, numerical_vector, game_status
    
def random_last_play(token_vector, numerical_vector, period, game_status, indices, token_model = None, match_vector = None):
    dict_tokens = define_special_tokens(config = 1)

    # quando nao ha mais tempo restante para gerar jogadas, precisamos forcar que uma jogada que 
    # encerre o periodo seja sorteada aqui nao precisa sortear tempo gasto porque ele sera o tempo 
    # remanescente da ultima jogada
    # se o periodo é o ultimo ou qualquer um do overtime, então não precisa sortear porque só tem 
    # um token que encerra periodos nessas condiçoes.
    if period == 1: only_consider_tokens = dict_tokens['end_period1']
    elif period == 2: only_consider_tokens = dict_tokens['end_period2']
    elif period == 3: only_consider_tokens = dict_tokens['end_period3']
    elif period == 4: only_consider_tokens = dict_tokens['end_period4']
    else: only_consider_tokens = dict_tokens['end_overtime']

    if isinstance(token_vector, list): token_vector = np.array(token_vector)
    if isinstance(numerical_vector, list): numerical_vector = np.array(numerical_vector)
    if not isinstance(indices, list): indices = [indices]

    if match_vector is None:
        numerical_vector_ = numerical_vector
    else:
        numerical_vector_ = np.hstack([numerical_vector, match_vector])
        
    random_token, prob_random_token = predict_model(model = token_model, token_vector = token_vector[indices], 
                                                    numerical_vector = numerical_vector_[indices], only_consider = only_consider_tokens)

    # quando estamos sorteando a ultima jogada nao precisamos sortear tempo gasto. 
    # Precisa apenas salvar o ultimo tempo restante e definir o tempo restante atual igual a 0
    temp_game_status = filter_elements(element = game_status, indices = indices)
    former_remaining_time = get_element_from_dict(dict_ = temp_game_status, key = 'remaining_time')
    update_element_in_game_status(game_status = game_status, indices = indices, key = 'remaining_time', new_value = 0)
    token_vector[indices], numerical_vector[indices] = update_variables(random_play = random_token, tokens = token_vector[indices], 
                                                                        game_status = filter_elements(element = game_status, indices = indices), 
                                                                        period = period)
    return random_token, prob_random_token, former_remaining_time, token_vector[indices], numerical_vector[indices], filter_elements(element = game_status, indices = indices)

def generate_period_plays(period, token_model, time_model, num_simulations = 100, token_vector =  [0] * 5, numerical_vector = [0.] * 14, 
                          game_status = None, game_log = None):
    dict_tokens = define_special_tokens(config = 1)
    if period == 1:
        # iniciando contextos:        
        token_vector = [token_vector] * num_simulations
        numerical_vector = [numerical_vector] * num_simulations
        token_vector = np.array(token_vector)
        numerical_vector = np.array(numerical_vector)

        # iniciando historia do jogo
        empty_list = [None] * 1000
        game_log_template = {'period': empty_list.copy(), 'token': empty_list.copy(), 'prob': empty_list.copy(), 
                             'time': empty_list.copy(), 'game_variables': empty_list.copy(), 
                             'token_variables': empty_list.copy(), 'numerical_variables': empty_list.copy(), 
                             'index': 0}
        
        game_log = [{k: v.copy() if isinstance(v, list) else v for k, v in game_log_template.items()} for _ in range(num_simulations)]

        # gerando a primeira jogada da partida
        results = random_first_play(token_vector = token_vector, numerical_vector = numerical_vector, 
                                    period = period, token_model = token_model)
        play, prob_play, _time, token_vector, numerical_vector, game_status_ = results
    else:
        token_vector = [game_log[i]['token_variables'][:game_log[i]['index']][-1] for i in range(num_simulations)]
        token_vector = torch.tensor(np.array(token_vector))
        numerical_vector = [game_log[i]['numerical_variables'][:game_log[i]['index']][-1] for i in range(num_simulations)]
        numerical_vector = torch.Tensor(np.array(numerical_vector))
        # gerando a primeira jogada da partida
        results = random_first_play(token_vector = token_vector, numerical_vector = numerical_vector, 
                                    period = period, token_model = token_model, game_status = game_status)
        play, prob_play, _time, token_vector, numerical_vector, game_status_ = results

    keep_generating = [1] * num_simulations
    spent_time = [0] * num_simulations
    period_duration = 7200 if period <= 4 else 3000

    # tokens que encerram o periodo
    end_period_tokens = dict_tokens['end_period']    
    # tokens especiais que fazem com que possamos sortear jogadas com o tempo restante = 0
    special_tokens = dict_tokens['special_plays_end']    
    shot_blocked_tokens = dict_tokens['shot_blocked_end']
    
    number_play = 1
    while sum([1 if k > 0 and k < 6 else 0 for k in keep_generating]) > 0:
        # PASSO 0: Atualiza o spent_time e o number_play
        number_play += 1
        temp_game_status = [dict(item) for item in game_status_]
        keep_generating = update_game_log(game_log = game_log, period = period, play = play, prob = prob_play,
                                          time = _time, game_status = temp_game_status, token_vector = token_vector, 
                                          numerical_vector = numerical_vector, keep_generating = keep_generating)

        # PASSO 2: Gera a próxima jogada
        results = random_play_loop(token_vector = token_vector, numerical_vector = numerical_vector, 
                                   game_status = game_status_, period = period, token_model = token_model, 
                                   time_model = time_model)
        play, prob_play, _time, temp_token_vector, temp_numerical_vector, temp_game_status = results
    
        # caso 2 onde um token de encerramento foi sorteado
        list_simulations = [i for i, (k, p) in enumerate(zip(keep_generating, play)) if p in end_period_tokens and k == 1]
        keep_generating = [2 if i in list_simulations else j for i, j in enumerate(keep_generating)]

        # caso 3 onde um token especial foi sorteado e o tempo de jogo ja passou a duracao do periodo
        list_simulations = [i for i, (st, k, t, p) in enumerate(zip(spent_time, keep_generating, _time, play)) \
                            if p in special_tokens and (st + t) >= period_duration and k == 1]
        if len(list_simulations) > 0:
            temp_game_status3 = filter_elements(element = temp_game_status, indices = list_simulations)
            list_old_remaining_time = get_element_from_dict(dict_ = temp_game_status3, key = 'remaining_time')
            list_new_remaining_time = [max(0, i) for i in list_old_remaining_time]
            update_element_in_game_status(game_status = temp_game_status, indices = list_simulations, 
                                          key = 'remaining_time', new_value = list_new_remaining_time)
            keep_generating = [3 if i in list_simulations else j for i, j in enumerate(keep_generating)]

        # caso 4 onde um token de bloqueio foi sorteado e o tempo de jogo ja passou a duracao do periodo
        list_simulations = [i for i, (st, k, t, p) in enumerate(zip(spent_time, keep_generating, _time, play)) \
                    if p in shot_blocked_tokens and (st + t) >= period_duration and k == 1]
        if len(list_simulations) > 0:
            temp_game_status4 = filter_elements(element = temp_game_status, indices = list_simulations)
            list_old_remaining_time = get_element_from_dict(dict_ = temp_game_status4, key = 'remaining_time')
            list_new_remaining_time = [max(0, i) for i in list_old_remaining_time]
            update_element_in_game_status(game_status = temp_game_status, indices = list_simulations, 
                                          key = 'remaining_time', new_value = list_new_remaining_time)
            keep_generating = [4 if i in list_simulations else j for i, j in enumerate(keep_generating)]

        # caso 5 onde um token de bloqueio foi sorteado na penultima jogoda e o tempo de jogo ja passou a duracao do periodo
        list_simulations = [i for i, k in enumerate(keep_generating) if k == 5]
        if len(list_simulations) > 0:
            temp_game_status5 = filter_elements(element = temp_game_status, indices = list_simulations)
            list_old_remaining_time = get_element_from_dict(dict_ = temp_game_status5, key = 'remaining_time')
            list_new_remaining_time = [max(0, i) for i in list_old_remaining_time]
            update_element_in_game_status(game_status = temp_game_status, indices = list_simulations, 
                                          key = 'remaining_time', new_value = list_new_remaining_time)

        # caso 7 onde o tempo de jogo ja passou a duracao do periodo e a proxima jogada precisa ser um encerramento
        list_simulations = [i for i, (st, k, t) in enumerate(zip(spent_time, keep_generating, _time)) \
                            if (st + t) >= period_duration and k == 1]          
        keep_generating = [7 if i in list_simulations else j for i, j in enumerate(keep_generating)]

        list_indices_to_update_game_status = [i for i, k in enumerate(keep_generating) if k > 0 and k < 6]

        temp_game_status_update = filter_elements(temp_game_status, list_indices_to_update_game_status)
        
        list_values = get_element_from_dict(dict_ = temp_game_status_update, key = 'home_points')
        update_element_in_game_status(game_status = game_status_, indices = list_indices_to_update_game_status, 
                                      key = 'home_points', new_value = list_values)
        
        list_values = get_element_from_dict(dict_ = temp_game_status_update, key = 'away_points')
        update_element_in_game_status(game_status = game_status_, indices = list_indices_to_update_game_status, 
                                      key = 'away_points', new_value = list_values)
    
        list_values = get_element_from_dict(dict_ = temp_game_status_update, key = 'limhteam')
        update_element_in_game_status(game_status = game_status_, indices = list_indices_to_update_game_status, 
                                      key = 'limhteam', new_value = list_values) 
    
        list_values = get_element_from_dict(dict_ = temp_game_status_update, key = 'limvteam')
        update_element_in_game_status(game_status = game_status_, indices = list_indices_to_update_game_status, 
                                      key = 'limvteam', new_value = list_values)
        
        list_values = get_element_from_dict(dict_ = temp_game_status_update, key = 'ball_possession')
        update_element_in_game_status(game_status = game_status_, indices = list_indices_to_update_game_status, 
                                      key = 'ball_possession', new_value = list_values) 
    
        list_values = get_element_from_dict(dict_ = temp_game_status_update, key = 'remaining_time')
        update_element_in_game_status(game_status = game_status_, indices = list_indices_to_update_game_status, 
                                      key = 'remaining_time', new_value = list_values) 

        _time = [period_duration - spent_time[k] if keep_generating[k] != 1 else _time[k] for k in range(len(spent_time))]
        spent_time = [spent_time[k] + _time[k] for k in range(len(spent_time))]
    
        token_vector[list_indices_to_update_game_status] = temp_token_vector[list_indices_to_update_game_status]
        numerical_vector[list_indices_to_update_game_status] = temp_numerical_vector[list_indices_to_update_game_status]
    
    indices_sort_end_period_play = [i for i, k in enumerate(keep_generating) if k > 0]
    if len(indices_sort_end_period_play) > 0:
        temp_token_vector = token_vector.copy()
        temp_numerical_vector = numerical_vector.copy()
        temp_game_status = [dict(item) for item in game_status_]

        play, prob_play, _time, temp_token_vector, temp_numerical_vector, temp_game_status = random_last_play(
            token_vector = temp_token_vector, numerical_vector = temp_numerical_vector, period = period, 
            game_status = temp_game_status, indices = indices_sort_end_period_play, token_model = token_model)

        list_values = get_element_from_dict(dict_ = temp_game_status, key = 'home_points')
        update_element_in_game_status(game_status = game_status_, indices = indices_sort_end_period_play, 
                                      key = 'home_points', new_value = list_values)

        list_values = get_element_from_dict(dict_ = temp_game_status, key = 'away_points')
        update_element_in_game_status(game_status = game_status_, indices = indices_sort_end_period_play, 
                                      key = 'away_points', new_value = list_values)

        list_values = get_element_from_dict(dict_ = temp_game_status, key = 'limhteam')
        update_element_in_game_status(game_status = game_status_, indices = indices_sort_end_period_play, 
                                      key = 'limhteam', new_value = list_values)

        list_values = get_element_from_dict(dict_ = temp_game_status, key = 'limvteam')
        update_element_in_game_status(game_status = game_status_, indices = indices_sort_end_period_play, 
                                      key = 'limvteam', new_value = list_values)

        list_values = get_element_from_dict(dict_ = temp_game_status, key = 'ball_possession')
        update_element_in_game_status(game_status = game_status_, indices = indices_sort_end_period_play, 
                                      key = 'ball_possession', new_value = list_values)

        list_values = get_element_from_dict(dict_ = temp_game_status, key = 'remaining_time')
        update_element_in_game_status(game_status = game_status_, indices = indices_sort_end_period_play, 
                                      key = 'remaining_time', new_value = list_values)
    
        token_vector[indices_sort_end_period_play] = temp_token_vector
        numerical_vector[indices_sort_end_period_play] = temp_numerical_vector

        update_game_log_by_indices(game_log = game_log, period = period, play = play, prob = prob_play, 
                                   time = _time, game_status = temp_game_status, token_vector = temp_token_vector, 
                                   numerical_vector = temp_numerical_vector, indices = indices_sort_end_period_play)

    # verifica se tem instant replay depois do fim do periodo, mas pela caracteristica dos dados, sorteia mais vezes do que essa situação acontece no mundo raal
    # results = random_play_loop(token_vector = token_vector, numerical_vector = numerical_vector, game_status = game_status_, 
    #                            period = period, token_model = token_model, time_model = time_model)
    # play, prob_play, _time, temp_token_vector, temp_numerical_vector, temp_game_status = results

    # indices_instant_review = [i for i, p in enumerate(play) if p in dict_tokens['instant_replay']]
    # token_vector[indices_instant_review] = temp_token_vector[indices_instant_review]

    # play = [p for i, p in enumerate(play) if i in indices_instant_review]
    # prob_play = [p for i, p in enumerate(prob_play) if i in indices_instant_review]
    # _time = [t for i, t in enumerate(_time) if i in indices_instant_review]

    # update_game_log_by_indices(game_log = game_log, period = period, play = play, prob = prob_play, time = _time,
    #                            game_status = filter_elements(temp_game_status, indices_instant_review),
    #                            token_vector = token_vector[indices_instant_review],
    #                            numerical_vector = numerical_vector[indices_instant_review], 
    #                            indices = indices_instant_review)

    return game_status_, game_log

def generate_game_plays(token_model, time_model, num_simulations = 100, verbose = 1, token_vector = [0] * 5, numerical_vector = [0.] * 14):

    df_partial_points = {}
    for period_ in range(1, 5):
        if period_ == 1:
            gstatus, glog = generate_period_plays(period = period_, token_model = token_model, 
                                                  time_model = time_model, num_simulations = num_simulations, 
                                                  token_vector = token_vector, numerical_vector = numerical_vector)
        else:
            gstatus, glog = generate_period_plays(period = period_, token_model = token_model,
                                                  time_model = time_model, num_simulations = num_simulations, 
                                                  game_status = gstatus, game_log = glog)
            
        df_partial_points[period_] = pd.DataFrame(gstatus)\
        .assign(period = period_)\
        .assign(outcome = lambda x: ['home' if h > a else 'away' if a > h else 'tie' for h, a in zip(x['home_points'], x['away_points'])])
        if verbose:
            print(f"Generated plays for period: {period_}. Here some metrics:")
            print(df_partial_points[period_][['home_points', 'away_points']].describe().T.round(2))
            print(df_partial_points[period_].groupby('outcome', as_index = False).size().set_index("outcome").T/num_simulations)
            
    indices_overtime = pd.DataFrame(gstatus).query("home_points == away_points").index.to_list()

    while len(indices_overtime) > 0:
        period_ += 1
        glog_overtime = filter_elements(element = glog, indices = indices_overtime)
        gstatus_overtime = filter_elements(element = gstatus, indices = indices_overtime)

        gstatus_overtime, glog_overtime = generate_period_plays(period = period_, token_model = token_model, 
                                                                time_model = time_model, num_simulations = len(indices_overtime), 
                                                                game_status = gstatus_overtime, game_log = glog_overtime)
        df_partial_points[period_] = pd.DataFrame(gstatus_overtime, index = indices_overtime)\
        .assign(period = period_)\
        .assign(outcome = lambda x: ['home' if h > a else 'away' if a > h else 'tie' for h, a in zip(x['home_points'], x['away_points'])])

        if verbose:
            print(f"Generated plays for period: {period_}. Here some metrics:")
            print(df_partial_points[period_][['home_points', 'away_points']].describe().T.round(2))
            print(df_partial_points[period_].groupby('outcome', as_index = False).size().set_index("outcome").T/len(indices_overtime))
                    
        indices_overtime = pd.DataFrame(gstatus).query("home_points == away_points").index.to_list()

    final_df = pd.concat([game_log_to_dataframe(gamelog).assign(simulation = i) for i, gamelog in enumerate(glog)])
    df_partial_points = pd.concat(df_partial_points, axis = 0)\
    .reset_index()\
    .rename(columns = {'level_1': 'simulation'})\
    .drop(columns = 'level_0')
    return final_df, df_partial_points

def generate_period_plays_extrainfo(period, token_model, time_model, match_vector, num_simulations = 100, token_vector =  [0] * 5, 
                                    numerical_vector = [0.] * 14, game_status = None, game_log = None):
    dict_tokens = define_special_tokens(config = 1)
    match_vector = [match_vector] * num_simulations
    if period == 1:
        # iniciando contextos:
        token_vector = [token_vector] * num_simulations
        numerical_vector = [numerical_vector] * num_simulations
        token_vector = np.array(token_vector)
        numerical_vector = np.array(numerical_vector)

        # iniciando historia do jogo
        empty_list = [None] * 1000
        game_log_template = {'period': empty_list.copy(), 'token': empty_list.copy(), 'prob': empty_list.copy(), 
                             'time': empty_list.copy(), 'game_variables': empty_list.copy(), 
                             'token_variables': empty_list.copy(), 'numerical_variables': empty_list.copy(), 
                             'index': 0}
        
        game_log = [{k: v.copy() if isinstance(v, list) else v for k, v in game_log_template.items()} for _ in range(num_simulations)]

        # gerando a primeira jogada da partida
        results = random_first_play(token_vector = token_vector, numerical_vector = numerical_vector, 
                                    period = period, token_model = token_model, match_vector = match_vector)
        play, prob_play, _time, token_vector, numerical_vector, game_status_ = results
    else:
        token_vector = [game_log[i]['token_variables'][:game_log[i]['index']][-1] for i in range(num_simulations)]
        token_vector = torch.tensor(np.array(token_vector))
        numerical_vector = [game_log[i]['numerical_variables'][:game_log[i]['index']][-1] for i in range(num_simulations)]
        numerical_vector = torch.Tensor(np.array(numerical_vector))
        # gerando a primeira jogada da partida
        results = random_first_play(token_vector = token_vector, numerical_vector = numerical_vector, 
                                    period = period, token_model = token_model, game_status = game_status, 
                                    match_vector = match_vector)
        play, prob_play, _time, token_vector, numerical_vector, game_status_ = results

    keep_generating = [1] * num_simulations
    spent_time = [0] * num_simulations
    period_duration = 7200 if period <= 4 else 3000

    # tokens que encerram o periodo
    end_period_tokens = dict_tokens['end_period']    
    # tokens especiais que fazem com que possamos sortear jogadas com o tempo restante = 0
    special_tokens = dict_tokens['special_plays_end']    
    shot_blocked_tokens = dict_tokens['shot_blocked_end']
    
    number_play = 1
    while sum([1 if k > 0 and k < 6 else 0 for k in keep_generating]) > 0:
        # PASSO 0: Atualiza o spent_time e o number_play
        number_play += 1
        temp_game_status = [dict(item) for item in game_status_]
        keep_generating = update_game_log(game_log = game_log, period = period, play = play, prob = prob_play,
                                          time = _time, game_status = temp_game_status, token_vector = token_vector, 
                                          numerical_vector = numerical_vector, keep_generating = keep_generating)

        # PASSO 2: Gera a próxima jogada
        results = random_play_loop(token_vector = token_vector, numerical_vector = numerical_vector, 
                                   game_status = game_status_, period = period, token_model = token_model, 
                                   time_model = time_model, match_vector = match_vector)
        play, prob_play, _time, temp_token_vector, temp_numerical_vector, temp_game_status = results
    
        # caso 2 onde um token de encerramento foi sorteado
        list_simulations = [i for i, (k, p) in enumerate(zip(keep_generating, play)) if p in end_period_tokens and k == 1]
        keep_generating = [2 if i in list_simulations else j for i, j in enumerate(keep_generating)]

        # caso 3 onde um token especial foi sorteado e o tempo de jogo ja passou a duracao do periodo
        list_simulations = [i for i, (st, k, t, p) in enumerate(zip(spent_time, keep_generating, _time, play)) \
                            if p in special_tokens and (st + t) >= period_duration and k == 1]
        if len(list_simulations) > 0:
            temp_game_status3 = filter_elements(element = temp_game_status, indices = list_simulations)
            list_old_remaining_time = get_element_from_dict(dict_ = temp_game_status3, key = 'remaining_time')
            list_new_remaining_time = [max(0, i) for i in list_old_remaining_time]
            update_element_in_game_status(game_status = temp_game_status, indices = list_simulations, 
                                          key = 'remaining_time', new_value = list_new_remaining_time)
            keep_generating = [3 if i in list_simulations else j for i, j in enumerate(keep_generating)]

        # caso 4 onde um token de bloqueio foi sorteado e o tempo de jogo ja passou a duracao do periodo
        list_simulations = [i for i, (st, k, t, p) in enumerate(zip(spent_time, keep_generating, _time, play)) \
                    if p in shot_blocked_tokens and (st + t) >= period_duration and k == 1]
        if len(list_simulations) > 0:
            temp_game_status4 = filter_elements(element = temp_game_status, indices = list_simulations)
            list_old_remaining_time = get_element_from_dict(dict_ = temp_game_status4, key = 'remaining_time')
            list_new_remaining_time = [max(0, i) for i in list_old_remaining_time]
            update_element_in_game_status(game_status = temp_game_status, indices = list_simulations, 
                                          key = 'remaining_time', new_value = list_new_remaining_time)
            keep_generating = [4 if i in list_simulations else j for i, j in enumerate(keep_generating)]

        # caso 5 onde um token de bloqueio foi sorteado na penultima jogoda e o tempo de jogo ja passou a duracao do periodo
        list_simulations = [i for i, k in enumerate(keep_generating) if k == 5]
        if len(list_simulations) > 0:
            temp_game_status5 = filter_elements(element = temp_game_status, indices = list_simulations)
            list_old_remaining_time = get_element_from_dict(dict_ = temp_game_status5, key = 'remaining_time')
            list_new_remaining_time = [max(0, i) for i in list_old_remaining_time]
            update_element_in_game_status(game_status = temp_game_status, indices = list_simulations, 
                                          key = 'remaining_time', new_value = list_new_remaining_time)

        # caso 7 onde o tempo de jogo ja passou a duracao do periodo e a proxima jogada precisa ser um encerramento
        list_simulations = [i for i, (st, k, t) in enumerate(zip(spent_time, keep_generating, _time)) \
                            if (st + t) >= period_duration and k == 1]          
        keep_generating = [7 if i in list_simulations else j for i, j in enumerate(keep_generating)]

        list_indices_to_update_game_status = [i for i, k in enumerate(keep_generating) if k > 0 and k < 6]

        temp_game_status_update = filter_elements(temp_game_status, list_indices_to_update_game_status)
        
        list_values = get_element_from_dict(dict_ = temp_game_status_update, key = 'home_points')
        update_element_in_game_status(game_status = game_status_, indices = list_indices_to_update_game_status, 
                                      key = 'home_points', new_value = list_values)
        
        list_values = get_element_from_dict(dict_ = temp_game_status_update, key = 'away_points')
        update_element_in_game_status(game_status = game_status_, indices = list_indices_to_update_game_status, 
                                      key = 'away_points', new_value = list_values)
    
        list_values = get_element_from_dict(dict_ = temp_game_status_update, key = 'limhteam')
        update_element_in_game_status(game_status = game_status_, indices = list_indices_to_update_game_status, 
                                      key = 'limhteam', new_value = list_values) 
    
        list_values = get_element_from_dict(dict_ = temp_game_status_update, key = 'limvteam')
        update_element_in_game_status(game_status = game_status_, indices = list_indices_to_update_game_status, 
                                      key = 'limvteam', new_value = list_values)
        
        list_values = get_element_from_dict(dict_ = temp_game_status_update, key = 'ball_possession')
        update_element_in_game_status(game_status = game_status_, indices = list_indices_to_update_game_status, 
                                      key = 'ball_possession', new_value = list_values) 
    
        list_values = get_element_from_dict(dict_ = temp_game_status_update, key = 'remaining_time')
        update_element_in_game_status(game_status = game_status_, indices = list_indices_to_update_game_status, 
                                      key = 'remaining_time', new_value = list_values) 

        _time = [period_duration - spent_time[k] if keep_generating[k] != 1 else _time[k] for k in range(len(spent_time))]
        spent_time = [spent_time[k] + _time[k] for k in range(len(spent_time))]
    
        token_vector[list_indices_to_update_game_status] = temp_token_vector[list_indices_to_update_game_status]
        numerical_vector[list_indices_to_update_game_status] = temp_numerical_vector[list_indices_to_update_game_status]
    
    indices_sort_end_period_play = [i for i, k in enumerate(keep_generating) if k > 0]
    if len(indices_sort_end_period_play) > 0:
        temp_token_vector = token_vector.copy()
        temp_numerical_vector = numerical_vector.copy()
        temp_game_status = [dict(item) for item in game_status_]

        play, prob_play, _time, temp_token_vector, temp_numerical_vector, temp_game_status = random_last_play(
            token_vector = temp_token_vector, numerical_vector = temp_numerical_vector, period = period, 
            game_status = temp_game_status, indices = indices_sort_end_period_play, token_model = token_model, 
            match_vector = match_vector)

        list_values = get_element_from_dict(dict_ = temp_game_status, key = 'home_points')
        update_element_in_game_status(game_status = game_status_, indices = indices_sort_end_period_play, 
                                      key = 'home_points', new_value = list_values)

        list_values = get_element_from_dict(dict_ = temp_game_status, key = 'away_points')
        update_element_in_game_status(game_status = game_status_, indices = indices_sort_end_period_play, 
                                      key = 'away_points', new_value = list_values)

        list_values = get_element_from_dict(dict_ = temp_game_status, key = 'limhteam')
        update_element_in_game_status(game_status = game_status_, indices = indices_sort_end_period_play, 
                                      key = 'limhteam', new_value = list_values)

        list_values = get_element_from_dict(dict_ = temp_game_status, key = 'limvteam')
        update_element_in_game_status(game_status = game_status_, indices = indices_sort_end_period_play, 
                                      key = 'limvteam', new_value = list_values)

        list_values = get_element_from_dict(dict_ = temp_game_status, key = 'ball_possession')
        update_element_in_game_status(game_status = game_status_, indices = indices_sort_end_period_play, 
                                      key = 'ball_possession', new_value = list_values)

        list_values = get_element_from_dict(dict_ = temp_game_status, key = 'remaining_time')
        update_element_in_game_status(game_status = game_status_, indices = indices_sort_end_period_play, 
                                      key = 'remaining_time', new_value = list_values)
    
        token_vector[indices_sort_end_period_play] = temp_token_vector
        numerical_vector[indices_sort_end_period_play] = temp_numerical_vector

        update_game_log_by_indices(game_log = game_log, period = period, play = play, prob = prob_play, 
                                   time = _time, game_status = temp_game_status, token_vector = temp_token_vector, 
                                   numerical_vector = temp_numerical_vector, indices = indices_sort_end_period_play)

    # verifica se tem instant replay depois do fim do periodo, mas pela caracteristica dos dados, sorteia mais vezes do que essa situação acontece no mundo raal
    # results = random_play_loop(token_vector = token_vector, numerical_vector = numerical_vector, game_status = game_status_, 
    #                            period = period, token_model = token_model, time_model = time_model, match_vector = match_vector)
    # play, prob_play, _time, temp_token_vector, temp_numerical_vector, temp_game_status = results

    # indices_instant_review = [i for i, p in enumerate(play) if p in dict_tokens['instant_replay']]
    # token_vector[indices_instant_review] = temp_token_vector[indices_instant_review]

    # play = [p for i, p in enumerate(play) if i in indices_instant_review]
    # prob_play = [p for i, p in enumerate(prob_play) if i in indices_instant_review]
    # _time = [t for i, t in enumerate(_time) if i in indices_instant_review]

    # update_game_log_by_indices(game_log = game_log, period = period, play = play, prob = prob_play, time = _time,
    #                            game_status = filter_elements(temp_game_status, indices_instant_review),
    #                            token_vector = token_vector[indices_instant_review],
    #                            numerical_vector = numerical_vector[indices_instant_review], 
    #                            indices = indices_instant_review)

    return game_status_, game_log

def generate_game_plays_extrainfo(token_model, time_model, match_vector, num_simulations = 100, verbose = 1, token_vector = [0] * 5, numerical_vector = [0.] * 14):

    df_partial_points = {}
    for period_ in range(1, 5):
        if period_ == 1:
            gstatus, glog = generate_period_plays_extrainfo(period = period_, token_model = token_model,
                                                            time_model = time_model, match_vector = match_vector, 
                                                            num_simulations = num_simulations, token_vector = token_vector, 
                                                            numerical_vector = numerical_vector)
        else:
            gstatus, glog = generate_period_plays_extrainfo(period = period_, token_model = token_model,
                                                            time_model = time_model, match_vector = match_vector,
                                                            num_simulations = num_simulations, game_status = gstatus, 
                                                            game_log = glog)
            
        df_partial_points[period_] = pd.DataFrame(gstatus)\
        .assign(period = period_)\
        .assign(outcome = lambda x: ['home' if h > a else 'away' if a > h else 'tie' for h, a in zip(x['home_points'], x['away_points'])])
        if verbose:
            print(f"Generated plays for period: {period_}. Here some metrics:")
            print(df_partial_points[period_][['home_points', 'away_points']].describe().T.round(2))
            print(df_partial_points[period_].groupby('outcome', as_index = False).size().set_index("outcome").T/num_simulations)
            
    indices_overtime = pd.DataFrame(gstatus).query("home_points == away_points").index.to_list()

    while len(indices_overtime) > 0:
        period_ += 1
        glog_overtime = filter_elements(element = glog, indices = indices_overtime)
        gstatus_overtime = filter_elements(element = gstatus, indices = indices_overtime)

        gstatus_overtime, glog_overtime = generate_period_plays_extrainfo(period = period_, token_model = token_model,
                                                                          time_model = time_model, match_vector = match_vector,
                                                                          num_simulations = len(indices_overtime),
                                                                          game_status = gstatus_overtime, game_log = glog_overtime)
        df_partial_points[period_] = pd.DataFrame(gstatus_overtime, index = indices_overtime)\
        .assign(period = period_)\
        .assign(outcome = lambda x: ['home' if h > a else 'away' if a > h else 'tie' for h, a in zip(x['home_points'], x['away_points'])])

        if verbose:
            print(f"Generated plays for period: {period_}. Here some metrics:")
            print(df_partial_points[period_][['home_points', 'away_points']].describe().T.round(2))
            print(df_partial_points[period_].groupby('outcome', as_index = False).size().set_index("outcome").T/len(indices_overtime))
                    
        indices_overtime = pd.DataFrame(gstatus).query("home_points == away_points").index.to_list()

    final_df = pd.concat([game_log_to_dataframe(gamelog).assign(simulation = i) for i, gamelog in enumerate(glog)])
    df_partial_points = pd.concat(df_partial_points, axis = 0)\
    .reset_index()\
    .rename(columns = {'level_1': 'simulation'})\
    .drop(columns = 'level_0')
    return final_df, df_partial_points

